# Learning to Dampen the Duffing Oscillator

In this notebook we will explore training a neural network to dampen a Simple Harmonic Oscillator

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from matplotlib import pyplot as plt

import desolver as de
import torch

Using `autoray` backend


## Specifying the Dynamical System

Now let's specify the right hand side of our dynamical system. It should be

$$
\ddot x + \delta\dot x + \alpha x + \beta x^3 = \gamma\cos(\omega t)
$$

But desolver only works with first order differential equations, thus we must cast this into a first order system before we can solve it. Thus we obtain the following system

$$
\begin{array}{l}
\frac{\mathrm{d}x}{\mathrm{dt}} = v_x \\
\frac{\mathrm{d}v_x}{\mathrm{dt}} = -\delta v_x - \alpha x - \beta x^3 + \gamma\cos(\omega t)
\end{array}
$$

In [2]:
@de.rhs_prettifier(
    equ_repr="[vx, -k*x/m]",
    md_repr=r"""
$$
\frac{\mathrm{d}\vec{x}}{\mathrm{dt}} = \begin{bmatrix}
\omega \\
\frac{g\sin\theta + \cos\theta\left\{\frac{-F_p + d_N\cos\theta}{m_T} + g d_N\right\}-\frac{\mu_p\theta^{(1)}}{m_p l}}{l\left\{\frac{4}{3}-\frac{m_p\cos\theta}{m_T}\left[\cos\theta - d_N\right]\right\}} \\
v \\
\frac{F_p - m_p l \theta^{(2)}\cos\theta - N_c d_N}{m_T}
\end{bmatrix}
$$
""")
def rhs(time, state, force, cart_mass, pole_mass, pole_length, gravity, fr_cart, fr_pole):
    """
    Implementation of the cart-pole system described in: https://coneural.org/florian/papers/05_cart_pole.pdf
    """
    theta, theta_dot, _, x_dot = state[..., 0], state[..., 1], state[..., 2], state[..., 3]

    dtheta = theta_dot
    dx = x_dot

    stheta, ctheta = theta.sin(), theta.cos()

    theta_dot_sq = theta_dot.square()
    total_mass = cart_mass + pole_mass * stheta.square()
    
    pole_moment_of_inertia = pole_mass * pole_length / 2
    
    sgn_xdot = torch.where(x_dot.abs() < 1e-7, torch.zeros_like(x_dot), torch.sign(x_dot))
    paren1_pre = -force - theta_dot_sq * pole_moment_of_inertia * (stheta + fr_cart * ctheta * sgn_xdot) + fr_cart * gravity * sgn_xdot

    dtheta_dot_common = gravity * stheta - (fr_pole * x_dot) / pole_moment_of_inertia
    dtheta_dot = dtheta_dot_common + ctheta * paren1_pre
    dtheta_dot = dtheta_dot * 2 / pole_length / (
                4.0 / 3.0 - pole_mass * ctheta / total_mass * (ctheta - fr_cart * sgn_xdot))

    cart_normal_force = total_mass * gravity - pole_moment_of_inertia * (dtheta_dot * stheta + theta_dot_sq * ctheta)

    sgn_xdot = torch.sign(cart_normal_force * x_dot)
    corr_needed = (sgn_xdot != torch.sign(x_dot)) & (fr_cart > 0.0)
    paren1_post = -force - pole_moment_of_inertia * theta_dot_sq * (stheta + fr_cart * ctheta * sgn_xdot) + fr_cart * gravity * sgn_xdot

    dtheta_dot = torch.where(corr_needed, (dtheta_dot_common + ctheta * paren1_post) * 2 / pole_length / (
                    4.0 / 3.0 - pole_mass * ctheta / total_mass * (ctheta - fr_cart * sgn_xdot)
                ), dtheta_dot)

    cart_normal_force = torch.where(corr_needed, (cart_mass + pole_mass) * gravity - pole_moment_of_inertia * (
                dtheta_dot * stheta + theta_dot_sq * ctheta), cart_normal_force)

    dx_dot = (force + pole_moment_of_inertia * (theta_dot_sq * stheta - dtheta_dot * ctheta) - fr_cart * cart_normal_force * sgn_xdot) / (cart_mass + pole_mass)

    return torch.stack([dtheta, dtheta_dot, dx, dx_dot], dim=-1)

In [3]:
print(rhs)
display(rhs)


$$
\frac{\mathrm{d}\vec{x}}{\mathrm{dt}} = \begin{bmatrix}
\omega \\
\frac{g\sin\theta + \cos\theta\left\{\frac{-F_p + d_N\cos\theta}{m_T} + g d_N\right\}-\frac{\mu_p\theta^{(1)}}{m_p l}}{l\left\{\frac{4}{3}-\frac{m_p\cos\theta}{m_T}\left[\cos\theta - d_N\right]\right\}} \\
v \\
\frac{F_p - m_p l \theta^{(2)}\cos\theta - N_c d_N}{m_T}
\end{bmatrix}
$$




$$
\frac{\mathrm{d}\vec{x}}{\mathrm{dt}} = \begin{bmatrix}
\omega \\
\frac{g\sin\theta + \cos\theta\left\{\frac{-F_p + d_N\cos\theta}{m_T} + g d_N\right\}-\frac{\mu_p\theta^{(1)}}{m_p l}}{l\left\{\frac{4}{3}-\frac{m_p\cos\theta}{m_T}\left[\cos\theta - d_N\right]\right\}} \\
v \\
\frac{F_p - m_p l \theta^{(2)}\cos\theta - N_c d_N}{m_T}
\end{bmatrix}
$$


Let's specify the initial conditions as well

In [4]:
y_init = torch.tensor([torch.pi/7, 0., 0., 0.], dtype=torch.float32)

And now we're ready to integrate!

## The Numerical Integration

We will use the same constants from Wikipedia as our constants where the forcing amplitude increases and all the other parameters stay constants.

In [5]:
#Let's define the fixed constants

constants = dict(
    force = 0.0,
    cart_mass = 1.0,
    pole_mass = 1.0,
    pole_length = 1.0,
    gravity = 9.8067,
    fr_cart = 0.0,
    fr_pole = 0.0,
)

# Initial and Final integration times
t0 = 0.0
tf = 4.0

In [6]:
a = de.OdeSystem(rhs, y0=y_init, dense_output=True, t=(t0, tf), dt=1e-3, atol=0.0, rtol=1e-4, constants={**constants})
a.method = "RK87"
a.integrate(eta=True)

  0%|          | 0/4000 [00:00<?, ?it/s]

## Plotting the State and Phase Portrait

In [7]:
# Times to evaluate the system at
frame_rate = 24  # frames/second
eval_times = torch.linspace(a.t[0], a.t[-1], int(frame_rate * (a.t[-1] - a.t[0])), device=a.y[-1].device, dtype=a.y[-1].dtype)

In [8]:
from matplotlib import animation, rc

# set to location of ffmpeg to get animations working
# For Linux or Mac
plt.rcParams['animation.ffmpeg_path'] = '/usr/bin/ffmpeg'

# For Windows
# plt.rcParams['animation.ffmpeg_path'] = 'C:\\ProgramData\\chocolatey\\bin\\ffmpeg.exe'

from IPython.display import HTML

In [9]:
%%capture

plt.ioff()
from matplotlib import gridspec

state_eval = a.sol(eval_times)
cart_position = torch.stack([
    state_eval[..., 2], torch.zeros_like(state_eval[..., 2])
], dim=-1)
pole_position = torch.stack([
    state_eval[..., 2] + constants['pole_length'] * torch.sin(state_eval[..., 0]),
    constants['pole_length'] * torch.cos(state_eval[..., 0])
], dim=-1)

def plot_cart_animation(cart_pos, pole_pos, system_states, pole_length=constants['pole_length']):
    fig = plt.figure(figsize=(20, 4))

    gs  = gridspec.GridSpec(1, 3, width_ratios=[3, 1, 1])
    ax0 = fig.add_subplot(gs[0])
    ax0.set_aspect(1)
    ax1 = fig.add_subplot(gs[1])
    ax1.set_aspect(1)
    ax2 = fig.add_subplot(gs[2])
    ax2.set_aspect(1)

    max_ylim = cart_pos[...,1].abs().max().item() + pole_length
    max_xlim = cart_pos[...,0].abs().max().item() + pole_length
    max_xlim = max(3*max_ylim, max_xlim)
    
    ax0.set_xlim(-max_xlim, max_xlim)
    ax0.set_ylim(-max_ylim, max_ylim)
    ax0.axhline(0.0, linestyle='--', linewidth=1.0, color='green')

    artists = []
    
    for idx, _ in enumerate(cart_pos):
        artists.append([
            ax0.scatter(cart_pos[idx,0], cart_pos[idx,1], marker='o', color='k', s=10.0),
            *ax0.plot([cart_pos[idx,0], pole_pos[idx,0]], [cart_pos[idx,1], pole_pos[idx,1]], linewidth=1.0, color='k'),
            *ax1.plot(system_states[:idx+1, 0]/(2*torch.pi), system_states[:idx+1, 1]/(2*torch.pi), color='blue'),
            ax1.scatter(system_states[idx, 0]/(2*torch.pi), system_states[idx, 1]/(2*torch.pi), marker='o', color='k', s=5.0),
            *ax2.plot(system_states[:idx+1, 2], system_states[:idx+1, 3], color='blue'),
            ax2.scatter(system_states[idx, 2], system_states[idx, 3], marker='o', color='k', s=5.0)
        ])
        
    ax0.set_xlim(-max_xlim, max_xlim)
    ax0.set_ylim(-max_ylim, max_ylim)
    ax0.set_xlabel(r"$x$")
    ax0.set_ylabel(r"$y$")

    max_phase_theta = system_states[...,:2].abs().max().item()/(2*torch.pi)

    ax1.set_xlim(-max_phase_theta, max_phase_theta)
    ax1.set_ylim(-max_phase_theta, max_phase_theta)
    ax1.set_xlabel(r"$\theta~[\mathrm{deg}]$")
    ax1.set_ylabel(r"$\dot \theta~[\mathrm{deg\;s^{-1}}]$")
    ax1.grid(which='major')

    max_phase_cart = system_states[...,2:].abs().max().item()

    ax2.set_xlim(-max_phase_cart, max_phase_cart)
    ax2.set_ylim(-max_phase_cart, max_phase_cart)
    ax2.set_xlabel(r"$x~[\mathrm{m}]$")
    ax2.set_ylabel(r"$\dot x~[\mathrm{m\;s^{-1}}]$")
    ax2.grid(which='major')
    plt.tight_layout()

    return animation.ArtistAnimation(fig=fig, artists=artists, interval=1000*(eval_times[1] - eval_times[0]).item())

ani = plot_cart_animation(cart_position, pole_position, state_eval)

rc('animation', html='html5')

In [10]:
display(ani)

## Defining a Simple Neural Network

Now we can define a simple neural network, in this case a feed-forward network (or a dense network), to dampen the oscillations of the system. Specifically, we will treat the network as providing some continuous force `F` which will be applied at every timestep assuming no lag in the controller nor any discretisation issues

In [11]:
max_force = 80.0 # N

def nn_rhs(t, state, nn_controller, cart_mass, pole_mass, pole_length, gravity, fr_cart, fr_pole, **kwargs):
    global max_force
    neural_network_force = torch.func.functional_call(nn_controller, kwargs, (state[...,:4],))
    neural_network_force = max_force * neural_network_force[...,0]
    base_dynamics_rhs = rhs(t, state[...,:4], neural_network_force, cart_mass, pole_mass, pole_length, gravity, fr_cart, fr_pole)
    return base_dynamics_rhs

def loss_augmented_nn_rhs(t, state, nn_controller, cart_mass, pole_mass, pole_length, gravity, fr_cart, fr_pole, **kwargs):
    base_dynamics_rhs = nn_rhs(t, state, nn_controller, cart_mass, pole_mass, pole_length, gravity, fr_cart, fr_pole, **kwargs)
    return torch.cat([
        base_dynamics_rhs,
        state.square().sum(dim=-1, keepdim=True)
    ], dim=-1)

As we would like to integrate the system differentiably we use `desolver.torch_ext.torch_solve_ivp` which handles the forward/backward AD passes memory efficiently.

Under the hood, `torch_solve_ivp` utilises the adjoint method for reverse-mode AD and the tangent linear method for forward-mode AD. Furthermore, with the way that it is written, `torch_solve_ivp` allows differentiating multiple times and therefore estimating higher order derivatives. As we're training a simple NN to dampen a driven oscillator with first-order gradient descent, we do not utilise these features in this notebook.

In [12]:
from desolver.torch_ext import torch_solve_ivp
import copy
import gc

In [ ]:
y_init_nn = y_init.detach().clone().to('cuda' if torch.cuda.is_available() else 'cpu', torch.float32)
y_init_nn = torch.cat([
    y_init_nn,
    torch.zeros_like(y_init_nn[...,:1])
], dim=-1)
kick_vars_mask = torch.tensor([0, 1, 0, 1], dtype=torch.bool, device=y_init_nn.device)

state_dim = y_init_nn.shape[0] - 1
hidden_dim = 256
output_dim = 1

simple_nn = torch.nn.Sequential(
    torch.nn.Linear(state_dim, hidden_dim),
    torch.nn.SELU(),
    torch.nn.Linear(hidden_dim, hidden_dim),
    torch.nn.SELU(),
    torch.nn.Linear(hidden_dim, output_dim),
    torch.nn.Tanh()
).to('cuda' if torch.cuda.is_available() else 'cpu', y_init_nn.dtype)

def weights_init(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight, gain=0.5)
        if hasattr(m, "bias"):
            torch.nn.init.constant_(m.bias, 0.01)
    elif isinstance(m, torch.nn.PReLU):
        torch.nn.init.uniform_(m.weight)
        
simple_nn.apply(weights_init)

constants_no_force = copy.deepcopy(constants)
del constants_no_force["force"]

params = dict(simple_nn.named_parameters())
optimizer = torch.optim.NAdam(params.values(), lr=1e-3, weight_decay=1e-3, decoupled_weight_decay=True)

time_horizon = 0.2
max_integration_time = time_horizon
number_of_steps = 256
time_increase_interval = 32
batch_size = 16

def fell_below_angle_event(t, y, **kwargs):
    out = torch.pi/5 - torch.max(torch.abs(y[...,0]))
    return out

def went_too_far_event(t, y, **kwargs):
    out = 4.0 - torch.max(torch.abs(y[...,2]))
    return out

fell_below_angle_event.is_terminal = True
fell_below_angle_event.direction = 0
went_too_far_event.is_terminal = True
went_too_far_event.direction = 0

def straight_through_estimator(primal_path_value, grad_path_value):
    return grad_path_value + (primal_path_value - grad_path_value).detach()

atol = torch.zeros_like(y_init_nn)
rtol = torch.ones_like(y_init_nn)*1e-4
rtol[...,-1] = atol[...,-1] = torch.inf

def generate_batch(torch_generator:torch.Generator):
    return torch.stack([
        torch.randn(batch_size, generator=torch_generator, device=y_init_nn.device, dtype=y_init_nn.dtype)*torch.pi/7,
        torch.randn(batch_size, generator=torch_generator, device=y_init_nn.device, dtype=y_init_nn.dtype)*torch.pi/32,
        torch.rand(batch_size, generator=torch_generator, device=y_init_nn.device, dtype=y_init_nn.dtype)*2.0,
        torch.rand(batch_size, generator=torch_generator, device=y_init_nn.device, dtype=y_init_nn.dtype)*1e-2,
        torch.zeros(batch_size, device=y_init_nn.device, dtype=y_init_nn.dtype)
    ], dim=-1)

generator = torch.Generator(device=y_init_nn.device).manual_seed(32)

vmapped_nn_rhs = torch.vmap(loss_augmented_nn_rhs, in_dims=tuple([None, 0, None]+[None]*6))

def batched_nn_rhs(t, state, nn_controller, cart_mass, pole_mass, pole_length, gravity, fr_cart, fr_pole, **kwargs):
    return vmapped_nn_rhs(t, state, nn_controller, cart_mass, pole_mass, pole_length, gravity, fr_cart, fr_pole, **kwargs)

def closure():
    global time_horizon, max_integration_time, generator
    optimizer.zero_grad()
    y_init_batch = generate_batch(generator)
    integrated_system = torch_solve_ivp(batched_nn_rhs, t_span=(t0, t0 + time_horizon), y0=y_init_batch, method='RK87', atol=atol, rtol=rtol, args=(simple_nn,), kwargs={**constants_no_force, **params}, events=[fell_below_angle_event, went_too_far_event], first_step=1e-3, min_step=4e-5, adjoint_atol=0.0, adjoint_rtol=1e-2, show_prog_bar=True)
    max_integration_time = integrated_system.t[-1].item()
    # The loss is the integrated error over the timespan.
    # This penalises the network for taking more time to dampen the system
    loss = 0.025 * integrated_system.y[..., -1, -1].sum() + 0.95 * integrated_system.y[..., :4, -1].square().sum()
    if loss.requires_grad:
        loss.backward()
    return loss

best_loss = torch.inf
best_params = copy.deepcopy(simple_nn.state_dict())

for step_idx in range(number_of_steps + time_increase_interval):
    loss = optimizer.step(closure).item()
    if loss < best_loss*0.995:
        best_loss = loss
        best_params = copy.deepcopy(simple_nn.state_dict())
    print(f"[{step_idx+1}/{number_of_steps}] - loss: {loss:.4e}, best_loss: {best_loss:.4e}, tf: {max_integration_time:>8.4f}/{time_horizon:<8.4f}")
    if step_idx % time_increase_interval == 0 and step_idx != 0:
        time_horizon = time_horizon + (step_idx // time_increase_interval) * (1.0 - time_horizon) / (number_of_steps // time_increase_interval)
        best_loss = torch.inf
    gc.collect()
    torch.cuda.empty_cache()

  0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/201 [00:00<?, ?it/s]

[1/256] - loss: 6.1836e+02, best_loss: 6.1836e+02, tf:   0.2000/0.2000  


  0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/201 [00:00<?, ?it/s]

[2/256] - loss: 3.1828e+01, best_loss: 3.1828e+01, tf:   0.2000/0.2000  


  0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/101 [00:00<?, ?it/s]

In [ ]:
simple_nn.load_state_dict(best_params)


with torch.inference_mode():
    a_nn = de.OdeSystem(nn_rhs, y0=y_init_nn[...,:4], dense_output=True, t=(t0, tf), constants={**constants_no_force, "nn_controller": simple_nn}, atol=0.0, rtol=1e-7, dt=1e-8)
    a_nn.method = "RK87"
    a_nn.integrate(eta=True)

  0%|          | 0/400000001 [00:00<?, ?it/s]

In [ ]:
simple_nn(state_eval[...,:4].cuda())

tensor([[0.0488],
        [0.0560],
        [0.0570],
        [0.0575],
        [0.0582],
        [0.0589],
        [0.0597],
        [0.0604],
        [0.0615],
        [0.0628],
        [0.0644],
        [0.0662],
        [0.0684],
        [0.0706],
        [0.0732],
        [0.0760],
        [0.0793],
        [0.0829],
        [0.0869],
        [0.0914],
        [0.0965],
        [0.1022],
        [0.1085],
        [0.1156],
        [0.1239],
        [0.1335],
        [0.1446],
        [0.1579],
        [0.1742],
        [0.1945],
        [0.2207],
        [0.2559],
        [0.3046],
        [0.3763],
        [0.4876],
        [0.6616],
        [0.8771],
        [0.9939],
        [0.9970],
        [0.9958],
        [0.9993],
        [0.9267],
        [0.5421],
        [0.3391],
        [0.3060],
        [0.3704],
        [0.5402],
        [0.8231],
        [0.9960],
        [0.9866],
        [0.9893],
        [0.9958],
        [0.7441],
        [0.4713],
        [0.4020],
        [0

In [ ]:
%%capture

state_eval = a_nn.sol(eval_times.cuda()).cpu()
cart_position = torch.stack([
    state_eval[..., 2], torch.zeros_like(state_eval[..., 2])
], dim=-1)
pole_position = torch.stack([
    state_eval[..., 2] + constants['pole_length'] * torch.sin(state_eval[..., 0]),
    constants['pole_length'] * torch.cos(state_eval[..., 0])
], dim=-1)

ani = plot_cart_animation(cart_position, pole_position, state_eval)

rc('animation', html='html5')

In [ ]:
display(ani)